# Feature Engineering — Deep Dive

## NovaPay Fraud Detection

Explicación detallada de cada variable creada, con visualizaciones y teoría.

## Índice
1. Eliminación de IDs
2. Temporales
3. Ratios financieros
4. Geográficas
5. Log-transformaciones
6. Codificación de categóricas
7. Interacciones
8. Anti-Data Leakage
9. Evaluación de impacto

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; from pathlib import Path
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt; import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
sys.path.append(str(Path.cwd().parent))
from model.feature_engineering import *
sns.set_style('whitegrid'); plt.rcParams['figure.figsize'] = (14, 6)
CF = '#E53935'; CN = '#4CAF50'
df = pd.read_csv(Path.cwd().parent / 'Notebooks' / 'data' / 'dataset_fraude.csv')
print(f'{df.shape[0]} txns, {df.shape[1]} cols, {df["IS_FRAUD"].mean()*100:.2f}% fraude')
# Eliminar el otro target para evitar leakage en análisis
df.drop(columns=['IMPACTO_FRAUDE'], inplace=True, errors='ignore')

---
## 1. Eliminación de IDs

**11 columnas eliminadas** por alta cardinalidad, riesgo de memorización y data leakage.

In [ ]:
for c in ['id_cliente','id_cuenta','id_tarjeta','id_transaccion','cuenta_origen','cuenta_destino','direccion_ip_origen','identificador_dispositivo_fingerprint']:
    print(f'  {c:40s} → {df[c].nunique():5d} únicos ({df[c].nunique()/len(df)*100:.1f}%)')

---
## 2. Temporales

**hour, day_of_week, is_weekday** — Patrones circadianos y semanales del fraude.

In [ ]:
df_plot = df.copy()
df_plot['hour'] = pd.to_datetime(df_plot['fecha_hora']).dt.hour
df_plot['dow'] = pd.to_datetime(df_plot['fecha_hora']).dt.dayofweek
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
h = df_plot.groupby('hour')['IS_FRAUD'].mean()*100; c = df_plot.groupby('hour').size()
ax[0].bar(c.index, c.values, color='lightgray', alpha=0.7, label='Txns')
ax0 = ax[0].twinx(); ax0.plot(h.index, h.values, 'o-', color=CF, lw=2, label='% Fraude')
ax[0].set_title('Fraude por hora'); ax[0].set_ylabel('N', color='gray'); ax0.set_ylabel('%', color=CF)
dow = df_plot.groupby('dow')['IS_FRAUD'].mean()*100; dc = df_plot.groupby('dow').size()
ax[1].bar(['L','M','X','J','V','S','D'], dc.values, color='lightgray', alpha=0.7)
ax1 = ax[1].twinx(); ax1.plot(['L','M','X','J','V','S','D'], dow.values, 's-', color=CF, lw=2)
ax[1].set_title('Fraude por día'); plt.tight_layout(); plt.show()

---
## 3. Ratios financieros

**8 ratios** que normalizan respecto al perfil del cliente.

In [ ]:
def _safe_ratio(a, b):
    """Safe division: return a/b, handling zeros."""
    return np.where(b == 0, 0.0, a / b)


In [ ]:
df_fe = df.copy()
df_fe['txn_vs_limit_pct'] = df_fe['importe_transaccion'] / df_fe['limite_importe_transacciones'].replace(0, 1)
df_fe['txn_vs_monthly_avg_pct'] = _safe_ratio(df_fe['importe_transaccion'], df_fe['importe_medio_mensual'])
df_fe['outflow_inflow_ratio'] = _safe_ratio(df_fe['volumen_saliente_30_dias'], df_fe['volumen_entrante_30_dias'])
df_fe['net_flow_30d'] = df_fe['volumen_entrante_30_dias'] - df_fe['volumen_saliente_30_dias']
for col, label in [('txn_vs_limit_pct', 'txn/limite'), ('txn_vs_monthly_avg_pct', 'txn/media_mensual')]:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    upper = df_fe[col].quantile(0.99); d = df_fe[col].clip(upper=upper)
    bp = ax[0].boxplot([d[df_fe['IS_FRAUD']==0], d[df_fe['IS_FRAUD']==1]], labels=['No','Fraude'], patch_artist=True)
    bp['boxes'][0].set_facecolor(CN); bp['boxes'][1].set_facecolor(CF)
    ax[0].set_title(f'{label} (p99)')
    bins = np.linspace(0, upper, 50)
    ax[1].hist(df_fe[df_fe['IS_FRAUD']==0][col].clip(upper=upper), bins=bins, alpha=0.6, density=True, color=CN)
    ax[1].hist(df_fe[df_fe['IS_FRAUD']==1][col].clip(upper=upper), bins=bins, alpha=0.6, density=True, color=CF)
    ax[1].set_title('Distribución comparada')
    plt.tight_layout(); plt.show()
    s, p = stats.ks_2samp(df_fe[df_fe['IS_FRAUD']==0][col], df_fe[df_fe['IS_FRAUD']==1][col])
    med0, med1 = df_fe[df_fe['IS_FRAUD']==0][col].median(), df_fe[df_fe['IS_FRAUD']==1][col].median()
    print(f'{label}: med(no)={med0:.3f} med(fraude)={med1:.3f}  KS={s:.3f} p={p:.2e}')

---
## 4. Geográficas

**cross_border, cross_region, same_country_device** — inconsistencias de ubicación.

In [ ]:
df_fe['cross_border'] = (df_fe['customer_country'] != df_fe['operacion_pais']).astype(int)
df_fe['cross_region'] = (df_fe['customer_region'] != df_fe['operacion_region']).astype(int)
df_fe['same_country_device'] = ((df_fe['customer_country'] == df_fe['operacion_pais']) & (df['dispositivo_reconocido']==1)).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for i, v in enumerate(['cross_border','cross_region','same_country_device']):
    ct = pd.crosstab(df_fe[v], df_fe['IS_FRAUD'], normalize='index')*100
    ct.columns = ['No','Fraude']; ct.plot(kind='bar', ax=ax[i], color=[CN,CF], legend=False)
    ax[i].set_title(f'% por {v}'); ax[i].set_xticklabels(ax[i].get_xticklabels(), rotation=0)
ax[1].legend(); plt.tight_layout(); plt.show()
for v in ['cross_border','cross_region','same_country_device']:
    print(f'  {v}=0: {df_fe[df_fe[v]==0]["IS_FRAUD"].mean()*100:.2f}%  =1: {df_fe[df_fe[v]==1]["IS_FRAUD"].mean()*100:.2f}%')

---
## 5. Log-transformaciones

**log1p** reduce asimetría de distribuciones con colas pesadas.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 10))
for i, (o, _, l) in enumerate([('importe_transaccion','_','Importe'), ('saldo_actual','_','Saldo'), ('tiempo_desde_ultima_transaccion','_','Tiempo')]):
    upper = df[o].quantile(0.99)
    ax[0,i].hist(df[o].clip(upper=upper), bins=50, color='steelblue', edgecolor='white')
    ax[0,i].set_title(f'{l} (skew={df[o].skew():.1f})')
    ax[1,i].hist(np.log1p(df[o]), bins=50, color='coral', edgecolor='white')
    ax[1,i].set_title(f'{l} log1p (skew={np.log1p(df[o]).skew():.1f})')
plt.tight_layout(); plt.show()

---
## 6. Codificación de categóricas

**Frequency + Target encoding** comprimen información sin one-hot explosion.

In [ ]:
for col in ['tipo_cliente','metodo_autenticacion','estado_cuenta']:
    s = df.groupby(col).agg(cnt=('IS_FRAUD','count'), fr=('IS_FRAUD','mean')).sort_values('cnt', ascending=False)
    s['fr'] *= 100
    fig, ax = plt.subplots(1, 2, figsize=(14, max(4, len(s)*0.3)))
    ax[0].barh(s.index, s['cnt'], color='steelblue'); ax[0].set_title(f'Freq - {col}')
    colors = [CF if v > df['IS_FRAUD'].mean()*100 else CN for v in s['fr']]
    ax[1].barh(s.index, s['fr'], color=colors)
    ax[1].axvline(df['IS_FRAUD'].mean()*100, color='gray', ls='--', label=f'Media {df["IS_FRAUD"].mean()*100:.1f}%')
    ax[1].set_title(f'Target - {col}'); ax[1].legend()
    plt.tight_layout(); plt.show()

---
## 7. Interacciones

**txn_severity, txn_intensity, tenure_years** capturan comportamientos compuestos.

In [ ]:
df_fe['txn_intensity'] = df_fe['numero_transacciones_ultima_hora'] / (df_fe['tiempo_desde_ultima_transaccion'] + 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
upper = df_fe['txn_intensity'].quantile(0.99); d = df_fe['txn_intensity'].clip(upper=upper)
bp = ax[0].boxplot([d[df_fe['IS_FRAUD']==0], d[df_fe['IS_FRAUD']==1]], labels=['No','Fraude'], patch_artist=True)
bp['boxes'][0].set_facecolor(CN); bp['boxes'][1].set_facecolor(CF)
ax[0].set_title('Intensidad (p99)')
sample = df_fe.sample(500, random_state=42)
ax[1].scatter(sample['numero_transacciones_ultima_hora'], sample['importe_transaccion'].clip(upper=sample['importe_transaccion'].quantile(0.99)),
              c=sample['IS_FRAUD'], cmap='RdYlGn_r', alpha=0.6)
ax[1].set_xlabel('Txns última hora'); ax[1].set_ylabel('Importe (p99)')
plt.tight_layout(); plt.show()

---
## 8. Anti-Data Leakage

**Regla de oro:** todo fit estadístico solo con train.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
tr, te = prepare_train_test_split(df, 'IS_FRAUD')
fe_leak = FeatureEngineer(encode_target='IS_FRAUD'); fe_leak.fit(df)
fe_ok = FeatureEngineer(encode_target='IS_FRAUD'); fe_ok.fit(tr)
for label, fe in [('CON leakage', fe_leak), ('SIN leakage', fe_ok)]:
    Xtr = fe.transform(tr); Xte = fe.transform(te)
    ytr, yte = Xtr.pop('IS_FRAUD').values, Xte.pop('IS_FRAUD').values
    m = RandomForestClassifier(100, random_state=42).fit(Xtr, ytr)
    print(f'  {label}: AUC={roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}')

---
## 9. Evaluación de impacto

**Baseline vs Full FE** — mejora significativa con feature engineering.

In [ ]:
tr, te = prepare_train_test_split(df, 'IS_FRAUD')
fe = FeatureEngineer(encode_target='IS_FRAUD')
Xtr = fe.fit_transform(tr); ytr = Xtr.pop('IS_FRAUD').values
Xte = fe.transform(te); yte = Xte.pop('IS_FRAUD').values
num = Xtr.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler(); Xtr[num] = scaler.fit_transform(Xtr[num]); Xte[num] = scaler.transform(Xte[num])
rf_full = RandomForestClassifier(200, random_state=42).fit(Xtr, ytr)
# Baseline (solo one-hot)
drop = ID_COLS + ['numero_fraudes_ultimo_ano','IS_FRAUD','IMPACTO_FRAUDE']
Xtr_b = tr.drop(columns=[c for c in drop if c in tr.columns], errors='ignore')
Xte_b = te.drop(columns=[c for c in drop if c in te.columns], errors='ignore')
for col in [c for c in CAT_COLS if c in Xtr_b.columns]:
    Xtr_b = pd.get_dummies(Xtr_b, columns=[col], drop_first=True)
    Xte_b = pd.get_dummies(Xte_b, columns=[col], drop_first=True)
Xte_b = Xte_b.reindex(columns=Xtr_b.columns, fill_value=0)
num_b = Xtr_b.select_dtypes(include=[np.number]).columns.tolist()
scaler_b = StandardScaler(); Xtr_b[num_b] = scaler_b.fit_transform(Xtr_b[num_b]); Xte_b[num_b] = scaler_b.transform(Xte_b[num_b])
m = RandomForestClassifier(200, random_state=42).fit(Xtr_b, tr['IS_FRAUD'])
for label, auc_val, auprc_val, nf in [
    ('Baseline', roc_auc_score(te['IS_FRAUD'], m.predict_proba(Xte_b)[:,1]),
     average_precision_score(te['IS_FRAUD'], m.predict_proba(Xte_b)[:,1]), Xtr_b.shape[1]),
    ('Full FE', roc_auc_score(yte, rf_full.predict_proba(Xte)[:,1]),
     average_precision_score(yte, rf_full.predict_proba(Xte)[:,1]), Xtr.shape[1])]:
    print(f'  {label:10s} AUC={auc_val:.4f} AUPRC={auprc_val:.4f} Feats={nf}')

---
## Conclusiones

**~35 features** en 6 categorías:
- Temporales (3) | Ratios (8) | Geográficas (3) | Log (5) | Encoding (18) | Interacciones (3)

**Principios:** anti-leakage, normalización de escala, codificación compacta, interacciones explícitas.